In [2]:

import json
import os
from os import path

In [3]:

from typing import Any, Callable, TypedDict

Transformer = TypedDict('Transformer', {
    'validator': Callable[[Any], bool],
    'action': Callable[[Any], Any]
})

transfomers: list[Transformer] = [
    {
        'validator': lambda x: isinstance(x, dict) and len(x) == 1 and 'item' in x,
        'action': lambda x: x['item']
    },
    {
        'validator': lambda x: isinstance(x, dict) and len(x) == 1 and 'tag' in x,
        'action': lambda x: '#' + x['tag']
    },
]

def transform(o):
    for transformer in transfomers:
        if transformer['validator'](o):
            return transformer['action'](o)
    if type(o) == dict:
        return {k: transform(v) for k, v in o.items()}
    elif type(o) == list:
        return [transform(e) for e in o]
    return o


In [4]:
cwd = os.getcwd()
recipesFolder = path.join(cwd, 'crafting')

outFolder = path.join(cwd, 'out')
for dirPath, dirNames, fileNames in os.walk(recipesFolder):
    if not path.exists(outFolder):
        os.mkdir(outFolder)
    for name in fileNames:
        filePath = path.join(dirPath, name)
        with open(filePath) as f:
            loaded = json.load(f)
        transformed = transform(loaded)
        with open(path.join(outFolder, filePath), '+w') as f:
            json.dump(transformed, f, indent=2)
    pass
